# Zero Hour Atlas
## Satellite-Derived Mapping of Early Warning Last-Mile Reach Gaps in the Jamuna-Padma Char Corridor, Bangladesh

**RIMES Mapathon 2026 — Theme 01: Geospatial Data for Disaster Risk Assessment**

---

This notebook documents the complete analytical workflow for the Zero Hour Atlas project, including:
- Study area definition and boundary extraction
- Flood hazard probability mapping via Sentinel-1 GRD on Google Earth Engine
- Physical Hazard Index (PHI) computation
- Access & Network Index (ANI) computation (tower proximity, signal density)
- Seasonal Reach Index (SRI) computation (flood-season isolation days)
- Zero Hour Index (ZHI) composite score calculation
- Settlement-level vulnerability scoring and classification
- Land change detection: erosion/accretion time series (2017–2025)
- Map export and visualization


## 0. Environment Setup

In [ ]:
# Core dependencies
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import rasterio
from rasterio.plot import show
from rasterio.mask import mask
from rasterio.transform import from_bounds
from shapely.geometry import Point, Polygon, box
from shapely.ops import unary_union
from scipy.spatial import cKDTree
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Google Earth Engine (requires authenticated GEE account)
# import ee
# ee.Authenticate()
# ee.Initialize(project='your-gee-project')

print('Environment ready.')

## 1. Study Area Definition

**Corridor:** Kazipur (Sirajganj) to Doulatdiya (Rajbari), Jamuna-Padma confluence  
**Bounding box:** lon 89.61–89.85, lat 23.83–24.82  
**Area:** 1,225.2 km²  
**Projection:** EPSG:3857 (Web Mercator) for all spatial outputs

In [ ]:
# Study area bounding box
BBOX_LON_MIN = 89.61
BBOX_LON_MAX = 89.85
BBOX_LAT_MIN = 23.83
BBOX_LAT_MAX = 24.82

# Load study boundary
study_boundary = gpd.read_file('data/study_boundary.geojson')
print(f'CRS: {study_boundary.crs}')
print(f'Bounds: {study_boundary.total_bounds}')
study_boundary.plot(figsize=(6, 8), color='lightblue', edgecolor='navy', linewidth=1.2)
plt.title('Zero Hour Atlas — Study Boundary\nKazipur to Doulatdiya, Jamuna-Padma Corridor')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Flood Hazard Probability Mapping (PHI)

**Source:** Sentinel-1 GRD, ~160 images (2017–2024) processed on Google Earth Engine  
**Method:** Otsu thresholding on VV backscatter → binary flood masks → temporal frequency composite  
**Output:** Per-pixel flood probability raster (0–1), resampled to 30 m resolution

```python
# GEE Script (run in GEE Code Editor or via ee Python API)
# var s1 = ee.ImageCollection('COPERNICUS/S1_GRD')
#   .filterBounds(studyArea)
#   .filterDate('2017-01-01', '2024-12-31')
#   .filter(ee.Filter.eq('instrumentMode', 'IW'))
#   .select('VV');
# var floodFreq = s1.map(otsuThreshold).mean();
```

In [ ]:
# Load settlement data with PHI scores
settlements = pd.read_csv('data/settlements_scored.csv')
print(f'Named settlements: {len(settlements)}')
print(settlements[['settlement_name', 'phi_score', 'ani_score', 'sri_score', 'zhi_score', 'zhi_class']].head(16))

## 3. Access & Network Index (ANI)

**Sources:** OpenCelliD Bangladesh Extract (2025), OSM road network  
**Components:**  
- Mobile tower proximity (km, inverse-distance weighted)  
- Network technology tier (2G=0.3, 3G=0.6, 4G=1.0)  
- Road accessibility score (distance to nearest paved road)

**Tower vulnerability:** 10 of 25 towers within 5 km flood-risk zone (40%)

In [ ]:
# Load tower data
towers = pd.read_csv('data/towers.csv')
print(f'Total towers: {len(towers)}')
print(f'Flood-risk towers (< 5 km from flood zone): {towers["flood_risk"].sum()}')
print(towers.groupby(['network_type', 'flood_risk']).size().unstack(fill_value=0))

## 4. Seasonal Reach Index (SRI)

**Method:** Per-settlement flood probability × 365 days → estimated annual isolation days  
**Threshold:** >0.5 daily flood probability = isolated (inaccessible by road/boat)  
**Mean isolation (Critical ZHI settlements):** 347 days/year

In [ ]:
# Annual flood calendar
annual_flood = pd.read_csv('data/annual_flood.csv')

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(annual_flood['year'], annual_flood['flood_area_km2'], color='#2166AC', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Year')
ax.set_ylabel('Flood Area (km²)')
ax.set_title('Annual Flood Extent — Jamuna-Padma Char Corridor (2017–2024)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Zero Hour Index (ZHI) Composite

$$\text{ZHI} = 0.35 \times \text{PHI} + 0.35 \times \text{ANI} + 0.30 \times \text{SRI}$$

**Weights:** Expert-elicited, aligned with Bangladesh NDRCC EW4All National Roadmap 2025–2030

| Class | ZHI Range | Description |
|---|---|---|
| EWS Connected | < 15 | Full early warning system coverage |
| Partial EWS | 15–35 | Partial EWS signal, some gaps |
| Moderate Gap | 35–55 | Moderate reach gap, seasonal disruption |
| Severe Isolation | 55–75 | Severe EWS reach gap |
| Critical Zero Hour | ≥ 75 | EWS signal does not reach — critical last-mile gap |

In [ ]:
# ZHI classification
def classify_zhi(zhi):
    if zhi < 15:   return 'EWS Connected'
    elif zhi < 35: return 'Partial EWS'
    elif zhi < 55: return 'Moderate Gap'
    elif zhi < 75: return 'Severe Isolation'
    else:          return 'Critical Zero Hour'

settlements['zhi_class_computed'] = settlements['zhi_score'].apply(classify_zhi)

# Summary
class_counts = settlements['zhi_class_computed'].value_counts()
print('=== ZHI Classification Summary ===')
print(class_counts)
print(f'\nCritical Zero Hour: {class_counts.get("Critical Zero Hour", 0)} / {len(settlements)} settlements')

In [ ]:
# Settlement-level ZHI ranking
zhi_palette = {
    'Critical Zero Hour': '#D73027',
    'Severe Isolation':   '#FC8D59',
    'Moderate Gap':       '#FEE090',
    'Partial EWS':        '#91BFDB',
    'EWS Connected':      '#1B2A4A'
}

fig, ax = plt.subplots(figsize=(10, 7))
colors = settlements['zhi_class_computed'].map(zhi_palette)
bars = ax.barh(settlements['settlement_name'], settlements['zhi_score'],
               color=colors, edgecolor='white', linewidth=0.4)
ax.axvline(75, color='#D73027', linestyle='--', linewidth=1, alpha=0.7, label='Critical ZHI ≥ 75')
ax.set_xlabel('Zero Hour Index (ZHI)')
ax.set_title('Settlement-Level ZHI Scores — Zero Hour Atlas', fontweight='bold')
patches = [mpatches.Patch(color=v, label=k) for k, v in zhi_palette.items()]
ax.legend(handles=patches, loc='lower right', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('charts/ZHI_Settlement_Ranking.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Land Change Detection: Erosion & Accretion (2017–2025)

**Method:** Sentinel-2 NDWI thresholding → binary water masks → multi-year change detection  
**Period:** 2018–2025 (8 years, annual composites)  
**Key findings:**
- Total erosion: **43.5 km²** (2017–2025)
- Total accretion: **11.4 km²**
- Net land loss: **−32.1 km²**
- Erosion acceleration: 1.3 km²/yr (2018) → 9.8 km²/yr (2025) = **7.7× increase**

In [ ]:
# Erosion / accretion time series
erosion_df = pd.read_csv('data/erosion_accretion.csv')
print(erosion_df)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(erosion_df['year'].astype(str), -erosion_df['erosion_km2'],
        color='#D73027', label='Erosion', height=0.4)
ax.barh(erosion_df['year'].astype(str), erosion_df['accretion_km2'],
        color='#1A9850', label='Accretion', height=0.4)
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Area (km²)')
ax.set_title('Annual Erosion & Accretion — Jamuna-Padma Char Corridor (2018–2025)', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 7. Summary Statistics

In [ ]:
# Load and display summary statistics
summary = pd.read_csv('data/summary_statistics.csv')
print('=== Zero Hour Atlas — Summary Statistics ===')
for _, row in summary.iterrows():
    print(f'{row["metric"]:45s}: {row["value"]}')

## 8. Data Sources

| Dataset | Source | Period |
|---|---|---|
| Sentinel-1 GRD SAR | Copernicus / GEE | 2017–2024 |
| Sentinel-2 MSI (NDWI) | Copernicus / GEE | 2017–2025 |
| SRTM 30m DEM | NASA / USGS | 2000 (static) |
| Cell tower locations | OpenCelliD | 2025 |
| Administrative boundaries | GADM v4.1 | 2024 |
| Road network | OpenStreetMap | 2025 |
| Settlement points | OSM + field-verified | 2025 |
| Global Flood Monitoring | Copernicus EMS | 2024 |

---

**References:**
1. RIMES (2025). Impact-Based Forecasting Toolkit for Bangladesh.
2. Bangladesh NDRCC (2025). EW4All National Roadmap 2025–2030.
3. Takagi et al. (2023). Nat. Hazards Earth Syst. Sci., 23, 1–18.
4. Islam et al. (2022). Remote Sens., 14(8), 1823.
5. Biggs et al. (2021). Environ. Sci. Policy, 115, 74–83.
6. Steckler et al. (2022). J. Geophys. Res. Earth Surf., 127.
7. UNDP Bangladesh (2025). Climate Vulnerability Index: FY2025–26.
8. Copernicus EMS (2024). Global Flood Monitoring Technical Note v2.0.
9. OpenCelliD (2025). Cell Tower Database — Bangladesh Extract.